In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import json
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from typing import List, Dict, Any
import os
from openai import OpenAI
import time

In [ ]:
#Запрашиваем ссылку на обьявление avito
url = input('Ссылка на объявление Avito: ').strip()

# Запрашиваем ключ один раз
GEMINI_API_KEY = input("Введите ваш Google Gemini API ключ: ").strip()

print("✅ Ссылка на обьявление Avito получена")
print("✅ Gemini API ключ получен")

In [ ]:
# Загрузка словаря микрокатегорий
def load_mc_mapping(json_path='key_phrases.json'):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        # Создаём словарь {mcTitle: mcId}
        return {item['mcTitle']: item['mcId'] for item in data}
    except FileNotFoundError:
        print(f'Ошибка: файл {json_path} не найден')
        return {}
    except json.JSONDecodeError:
        print('Ошибка: некорректный JSON')
        return {}

# Инициализация сессии
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.avito.ru/',
}

session = requests.Session()
session.cookies.set('_ga', 'dummy')
session.cookies.set('_ym_uid', 'dummy')

try:
    resp = session.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # --- Парсинг ID ---
    id_span = soup.find('span', {'data-marker': 'item-view/item-id'})
    if id_span:
        text = id_span.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        item_id = int(match.group(1)) if match else None
    else:
        item_id = None

    # --- Парсинг описания ---
    desc_div = soup.find('div', {'data-marker': 'item-view/item-description'})
    if desc_div:
        description = ' '.join(desc_div.stripped_strings)
    else:
        description = 'Описание не найдено'

    # --- Парсинг микрокатегории (название) ---
    # Ищем все элементы breadcrumb с itemprop="name"
    name_spans = soup.find_all('span', {'itemprop': 'name'})
    mc_title = None
    if name_spans:
        # Последний элемент — текущая категория
        mc_title = name_spans[-1].get_text(strip=True)
    else:
        # Альтернативный поиск через data-marker (если структура иная)
        breadcrumb = soup.find('div', {'data-markers': 'breadcrumbs'})
        if breadcrumb:
            last_link = breadcrumb.find_all('a')[-1] if breadcrumb.find_all('a') else None
            if last_link:
                mc_title = last_link.get_text(strip=True)

    # --- Сопоставление с JSON ---
    mc_mapping = load_mc_mapping()  # загружаем соответствия
    if mc_title and mc_title in mc_mapping:
        mc_id = mc_mapping[mc_title]
    else:
        mc_id = "unknown"
        if not mc_title:
            mc_title = "Категория не найдена"

    # --- Итоговый словарь ---
    classified = {
        "itemId": item_id,
        "mcId": mc_id,
        "mcTitle": mc_title,
        "description": description
    }
    print(classified)

except Exception as e:
    print(f'Ошибка: {e}')

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Оставляем буквы, цифры, пробелы, базовые знаки препинания
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`_—-]'
    cleaned = re.sub(pattern, '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Применяем
description_raw = classified['description']
cleaned_description = clean_text(description_raw)

print(cleaned_description)

In [ ]:
# =============================================
# 1. Загрузка моделей и словаря (делается один раз при старте)
# =============================================

print("Загрузка моделей и токенизатора...")
tokenizer = AutoTokenizer.from_pretrained("deepvk/USER2-base")

# → Загружаем detected модель с auto-device (GPU если есть)
detected_model = AutoModelForSequenceClassification.from_pretrained(
    "./final_detected_model",
    device_map="auto"
)
detected_model.eval()

# → Загружаем split модель с auto-device
split_model = AutoModelForSequenceClassification.from_pretrained(
    "./final_split_model",
    device_map="auto"
)
split_model.eval()

# Пороги (из обучения)
DETECTED_THRESHOLD = 0.45
SPLIT_THRESHOLD = 0.25

# → Загружаем словарь микрокатегорий (mcId -> mcTitle)
with open("key_phrases.json", "r", encoding="utf-8") as f:
    key_phrases_list = json.load(f)  # список словарей

# Преобразуем в словарь для быстрого доступа
id_to_title = {item["mcId"]: item["mcTitle"] for item in key_phrases_list}
# Также нужен список всех возможных mcId для бинаризации (для idx2id)
all_mc_ids = sorted(id_to_title.keys())
id2idx = {mc_id: i for i, mc_id in enumerate(all_mc_ids)}
idx2id = {i: mc_id for mc_id, i in id2idx.items()}
print(f"Загружено {len(all_mc_ids)} микрокатегорий")

In [ ]:
# =============================================
# 2. Функция предсказания для одного текста
# =============================================

def predict_single(model, text: str, threshold: float) -> List[int]:
    """Возвращает список предсказанных mc_id для одного текста"""
    # → Автоматически определяем устройство модели
    device = next(model.parameters()).device
    
    encodings = tokenizer(
        [text],  # батч из одного элемента
        truncation=True,
        max_length=1024,
        padding=True,
        return_tensors="pt"
    )
    # → Переносим тензоры на устройство модели
    encodings = {k: v.to(device) for k, v in encodings.items()}
    
    with torch.no_grad():
        logits = model(**encodings).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # первый (и единственный) батч
    
    pred_ids = [idx2id[i] for i, p in enumerate(probs) if p > threshold]
    return pred_ids


In [ ]:
# =============================================
# 3. Основная функция обработки одного объявления
# =============================================

def process_single_ad(classified: Dict[str, Any], cleaned_description: str) -> Dict[str, Any]:
    """
    classified: словарь с полями itemId, mcId, mcTitle, description (последнее может быть грязным)
    cleaned_description: очищенное описание (строка)
    """
    # --- Шаг 1: Detected модель ---
    text_detected = f"classification: {cleaned_description}"
    detected_mc_ids = predict_single(detected_model, text_detected, DETECTED_THRESHOLD)
    
    # --- Шаг 2: Split модель (на основе предсказанных detected) ---
    if detected_mc_ids:
        detected_str = ", ".join(map(str, detected_mc_ids))
    else:
        detected_str = "none"
    text_split = f"categories: {detected_str} [SEP] {cleaned_description}"
    split_mc_ids = predict_single(split_model, text_split, SPLIT_THRESHOLD)
    
    # --- Шаг 3: Формируем результат ---
    should_split = len(split_mc_ids) > 0
    drafts = []
    for mc_id in split_mc_ids:
        title = id_to_title.get(mc_id, f"Категория {mc_id}")
        drafts.append({
            "mcId": mc_id,
            "mcTitle": title,
            "text": cleaned_description  # исходный текст, потом заменим через Gemini
        })
    
    result = {
        "detectedMcIds": detected_mc_ids,
        "shouldSplit": should_split,
        "drafts": drafts
    }
    return result

In [ ]:
# =============================================
# Функция генерации описания черновика через Gemini
# =============================================

# Создаём клиент
llm_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=GEMINI_API_KEY,
)

def generate_draft_description_text(mc_title: str, raw_draft_text: str) -> str:
    """Генерация через Gemini 3.1 Flash Lite Preview"""
    try:
        # Очень простой и короткий промпт — free-роутер это любит
        prompt = f"""Категория: {mc_title}

Текст: {raw_draft_text}

Извлеки только реальные услуги из этого текста, относящиеся к категории "{mc_title}". 
Ничего не придумывай. 
Если услуг мало или их сложно выделить — напиши минимальный вариант.

Ответ должен быть в формате:
Такие услуги как [перечисление через запятую] и другие, могут быть сделаны отдельно.
Ничего лишнего не говори, строго все по инструкции."""

        response = llm_client.chat.completions.create(
                model="gemini-3.1-flash-lite-preview",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=200,
                temperature=0.6,
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Ошибка Gemini API: {e}")
        return f"Услуги, связанные с {mc_title.lower()}, могут быть сделаны отдельно."

In [ ]:
# Финальная ячейка
draft_creator = process_single_ad(classified, cleaned_description)

if draft_creator.get("shouldSplit", False) and draft_creator.get("drafts"):
    print("🔄 Генерируем чистые описания для черновиков через Gemini...")
    
    for draft in draft_creator["drafts"]:
        mc_title = draft["mcTitle"]
        raw_text = draft["text"]
        
        clean_text = generate_draft_description_text(mc_title, raw_text)
        draft["text"] = clean_text
        print(f"   ✓ {mc_title} — готово")
    
    print("✅ Все описания черновиков сгенерированы")
else:
    print("ℹ️  shouldSplit = False — генерация не требуется")

# Красивый вывод
print("\n" + "="*70)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("="*70)
print(json.dumps(draft_creator, ensure_ascii=False, indent=2))